In [ ]:
# =============================================
# IMPORTY
# =============================================
import os
import sys
import math

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel



if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"✓ Device: {device}")
print(f"✓ PyTorch: {torch.__version__}")

# Nastav working directory na kořen projektu
_nb_dir = os.path.abspath('')
if os.path.basename(_nb_dir) in ('experiments', 'examples', 'train'):
    os.chdir(os.path.dirname(_nb_dir))
sys.path.insert(0, os.path.join(os.getcwd(), 'PFNs'))
print(f"✓ Working directory: {os.getcwd()}")
# PFNs — musí být naklonováno (viz setup.sh)
from pfns.train import train, MainConfig, OptimizerConfig, TransformerConfig, BatchShapeSamplerConfig
from pfns.model.encoders import EncoderConfig
from pfns.model.bar_distribution import BarDistributionConfig
from pfns.priors.prior import AdhocPriorConfig
from pfns.priors.fast_gp import get_batch as get_batch_for_gp

print("✓ PFNs načteny")

In [ ]:
# =============================================
# LOAD HELPER 
# =============================================

def load_pfn_model(checkpoint_path, device='cpu'):
    """
    Načte PFN model z checkpointu. Podporuje tři formáty:

    1. Nový formát (MainConfig dict, 'priors' key):
       - Uloženo přes epoch loop v PFN_TRAIN_SETUP.ipynb nebo pfn_train*.py
       - Config obsahuje celou architekturu → nlayers, emsize atd. se čtou automaticky

    2. Starý formát ('hps' key):
       - Uloženo přes train_gp_pfn()
       - Config obsahuje jen HP, architektura se musí rekonstruovat
       - nlayers se pokusí přečíst z config; pokud chybí, vyhodí chybu

    Vrací: (model, hps, epoch)
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Model nenalezen: {checkpoint_path}\n")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config_data = checkpoint.get('config', {})

    # --- Formát 1: MainConfig dict (nový) ---
    if isinstance(config_data, dict) and 'priors' in config_data:
        saved_config = MainConfig.from_dict(config_data)
        model = saved_config.model.create_model()
        model.load_state_dict(checkpoint['model_state_dict'])
        hps = saved_config.priors[0].prior_kwargs.get('hyperparameters', {})
        epoch = checkpoint.get('epoch', '?')

    # --- Formát 2: starý formát ('hps' key) ---
    elif isinstance(config_data, dict) and 'hps' in config_data:
        hps = config_data['hps']
        epoch = config_data.get('epochs', checkpoint.get('epoch', '?'))
        criterion = checkpoint['criterion']

        nlayers = config_data.get('nlayers')
        if nlayers is None:
            # Odvoď nlayers ze state_dict — spočítej nejvyšší index vrstvy
            layer_indices = [
                int(k.split('.')[2])
                for k in checkpoint['model_state_dict']
                if k.startswith('transformer_layers.layers.')
            ]
            nlayers = max(layer_indices) + 1 if layer_indices else 6

        emsize  = config_data.get('emsize',  512)
        nhead   = config_data.get('nhead',   8)
        nhid    = config_data.get('nhid',    1024)

        cfg = MainConfig(
            priors=[AdhocPriorConfig(
                get_batch_methods=[get_batch_for_gp],
                prior_kwargs={'num_features': 1, 'hyperparameters': hps}
            )],
            optimizer=OptimizerConfig('adamw', lr=0.0003),
            model=TransformerConfig(
                criterion=BarDistributionConfig(
                    full_support=True,
                    borders=criterion.borders.tolist()
                ),
                emsize=emsize, nhead=nhead, nhid=nhid, nlayers=nlayers,
                features_per_group=1,
                attention_between_features=False,
                encoder=EncoderConfig(
                    constant_normalization_mean=0.5,
                    constant_normalization_std=math.sqrt(1/12)
                )
            ),
            batch_shape_sampler=BatchShapeSamplerConfig(
                batch_size=64, max_seq_len=50,
                min_num_features=1, max_num_features=1
            ),
            epochs=1, warmup_epochs=0,
            steps_per_epoch=1, num_workers=0,
        )
        model = cfg.model.create_model()
        model.load_state_dict(checkpoint['model_state_dict'])
        model.criterion = criterion

    else:
        raise ValueError(f"Neznámý formát checkpointu. Klíče config: {list(config_data.keys())}")

    model.to(device)
    model.eval()
    print(f"✓ Model načten: {os.path.basename(checkpoint_path)}")
    print(f"  Epocha: {epoch}, nlayers: {nlayers if 'nlayers' in dir() else '(z MainConfig)'}")
    print(f"  Parametrů: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    return model, hps, epoch

print("✓ load_pfn_model připraven")

In [ ]:
# =============================================
# NAČTENÍ MODELŮ
# =============================================

# DŮLEŽITÉ: get_batch_for_gp_random_hps musí být definována PŘED torch.load
def get_batch_for_gp_random_hps(batch_size, seq_len, num_features, device="cpu", hyperparameters=None, **kwargs):
    random_hps = {
        "lengthscale": 0.3,
        "outputscale": 0.1,
        "noise":       1.0,
    }
    return get_batch_for_gp(batch_size, seq_len, num_features, device=device, hyperparameters=random_hps, **kwargs)


def load_for_inference(checkpoint_path, device='cpu'):
    """
    Načte libovolný PFN checkpoint pro inferenci.
    Automaticky detekuje nlayers z checkpointu — funguje pro 1L, 2L, 4L, 6L, ...
    """
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint["config"]

    if "num_features" in config:
        # Starý formát
        num_features     = config["num_features"]
        max_dataset_size = config["max_dataset_size"]
        criterion        = checkpoint["criterion"]
        borders          = criterion.borders.tolist()
        nlayers          = config.get("nlayers", 6)
        get_batch_fn     = get_batch_for_gp
        prior_kwargs     = {"num_features": num_features, "hyperparameters": config.get("hps", {})}
    else:
        # Nový formát (MainConfig)
        num_features     = config["priors"][0]["prior_kwargs"]["num_features"]
        max_dataset_size = config["batch_shape_sampler"]["max_seq_len"]
        borders          = config["model"]["criterion"]["borders"]
        nlayers          = config["model"].get("nlayers", 6)
        get_batch_fn     = get_batch_for_gp_random_hps
        prior_kwargs     = {"num_features": num_features, "hyperparameters": {}}
        criterion        = None

    model_config = MainConfig(
        priors=[AdhocPriorConfig(
            get_batch_methods=[get_batch_fn],
            prior_kwargs=prior_kwargs
        )],
        optimizer=OptimizerConfig("adamw", lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512, nhead=8, nhid=1024, nlayers=nlayers,
            features_per_group=1,
            attention_between_features=False,
            encoder=EncoderConfig(
                constant_normalization_mean=0.5,
                constant_normalization_std=math.sqrt(1/12)
            )
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=2, max_seq_len=max_dataset_size,
            min_num_features=num_features, max_num_features=num_features
        ),
        epochs=1, steps_per_epoch=1, num_workers=0,
    )

    dummy_result = train(model_config, device=device, reusable_config=False)
    model = dummy_result["model"]
    model.load_state_dict(checkpoint["model_state_dict"])
    if criterion is not None:
        model.criterion = criterion
    model.to(device)
    model.eval()

    epoch = checkpoint.get("epoch", "?")
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  ✓ {os.path.basename(checkpoint_path)}: nlayers={nlayers}, epoch={epoch}, {n_params:.2f}M params")
    return model, epoch


# =============================================
# KONFIGURACE MODELŮ — uprav cesty dle potřeby
# =============================================
MODEL_PATHS = {
    "1-layer": os.path.join("models", "pfn_rand_hps_1layer.pth"),
    "2-layer": os.path.join("models", "pfn_rand_hps_2layer.pth"),
    "4-layer": os.path.join("models", "pfn_rand_hps_4layer.pth"),
    "6-layer": os.path.join("models", "pfn_rand_hps_6layer.pth"),
    "8-layer": os.path.join("models", "pfn_rand_hps_8layer.pth"),
}

print("Načítám modely...")
MODELS = {}
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        MODELS[name], _ = load_for_inference(path, device)
    else:
        print(f"  ⚠ Nenalezeno: {path}")

# Zpětná kompatibilita — loaded_model = první dostupný model
loaded_model = next(iter(MODELS.values())) if MODELS else None

# =============================================
# GLOBÁLNÍ NASTAVENÍ
# =============================================
hps = {"noise": 1e-3, "outputscale": 1.0, "lengthscale": 0.3}

def reset_seed():
    import random
    s = random.randint(0, 2**32 - 1)
    torch.manual_seed(s)
    np.random.seed(s % (2**31))
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def get_batch_cpu(batch_size, seq_len, num_features, hyperparameters, device):
    batch = get_batch_for_gp(batch_size, seq_len, num_features,
                              device="cpu", hyperparameters=hyperparameters)
    from pfns.priors.prior import Batch
    return Batch(x=batch.x.to(device), y=batch.y.to(device), target_y=batch.target_y.to(device))

reset_seed()
print(f"\n✓ Načteno {len(MODELS)} modelů: {list(MODELS.keys())}")

## Experiment 4: Hyperparametrická nejistota a prediktivní distribuce

### Q4 — Crossover jako funkce šíře prioru (Step 5 z PFN-GP2.md)

PFN je meta-trénován přes prior $p(\theta)$. Při malém $n$ je posterior $p(\theta|X,y)$ plochý — ML-II nemá dost dat a přefituje $\theta$. PFN nese prior z meta-tréninku → lepší predikce.

**Crossover $n^*$:** kde NLL(PFN) = NLL(GP-ML). Závisí na šíři prioru:
- Úzký prior ($\ell \sim \text{LN}(-1, 0.2^2)$): ML-II identifikuje $\theta$ rychle → malé $n^*$
- Střední prior (existující modely, $\ell \sim U(0.05, 1.0)$): střední $n^*$
- Široký prior ($\ell \sim \text{LN}(-1, 1.5^2)$): ML-II potřebuje hodně bodů → velké $n^*$

---

### Q5 — Tvar prediktivní distribuce pod hyperparametrickou nejistotou (Step 6 z PFN-GP2.md)

Marginalizace přes $\theta$ dává **směs Gaussiánů**:
$$p(y_*|x_*, X, y) = \int \mathcal{N}(\mu_\theta, \sigma^2_\theta)\, p(\theta|X,y)\, d\theta$$

Pokud je $p(\theta|X,y)$ bimodální → prediktivní distribuce může být **bimodální nebo heavy-tailed**.
`BarDistribution` tuto distribuci reprezentovat umí.

**Test:** PFN natrénovaný na bimodálním prioru $\ell \sim 0.5\,N(0.1, 0.01^2) + 0.5\,N(0.8, 0.05^2)$.
Pro ambiguní kontext vizualizujeme BarDistribution vs. oracle směs $0.5\,\mathcal{N}_{0.1} + 0.5\,\mathcal{N}_{0.8}$.

In [ ]:
# =============================================
# PRIOR WRAPPERY PRO TRÉNOVÁNÍ
# =============================================
# Poznámka: get_batch_for_gp_random_hps je již definována v buňce 3 (NAČTENÍ MODELŮ)
# a odpovídá "střední" šíři prioru.

def get_batch_for_gp_narrow_prior(batch_size, seq_len, num_features,
                                   device='cpu', hyperparameters=None, **kwargs):
    """Úzký prior: ℓ ~ LN(-1, 0.2²) ≈ soustředěno kolem ℓ≈0.37"""
    ell = max(0.01, float(np.random.lognormal(-1.0, 0.2)))
    random_hps = {"lengthscale": ell, "outputscale": float(np.random.lognormal(0, 0.5)),
                  "noise": float(10**np.random.uniform(-3, -1))}
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def get_batch_for_gp_wide_prior(batch_size, seq_len, num_features,
                                 device='cpu', hyperparameters=None, **kwargs):
    """Široký prior: ℓ ~ LN(-1, 1.5²) ~ ℓ ∈ [0.01, 10]"""
    ell = max(0.01, float(np.random.lognormal(-1.0, 1.5)))
    random_hps = {"lengthscale": ell, "outputscale": float(np.random.lognormal(0, 0.5)),
                  "noise": float(10**np.random.uniform(-3, -1))}
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def get_batch_for_gp_bimodal_prior(batch_size, seq_len, num_features,
                                    device='cpu', hyperparameters=None, **kwargs):
    """Bimodální prior: ℓ ~ 0.5·N(0.1, 0.01²) + 0.5·N(0.8, 0.05²)"""
    ell = float(np.random.normal(0.1, 0.01)) if np.random.random() < 0.5 else float(np.random.normal(0.8, 0.05))
    ell = max(0.005, ell)
    random_hps = {"lengthscale": ell, "outputscale": 1.0, "noise": 0.01}
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)

print("✓ Prior wrappery připraveny")

In [ ]:
# =============================================
# TRÉNOVACÍ FUNKCE (přepoužitelná)
# =============================================
from pfns.model import bar_distribution as bar_dist_module

def train_pfn_with_prior(get_batch_fn, save_path, epochs=300,
                          num_features=1, max_dataset_size=128,
                          batch_size=64, steps_per_epoch=100, nlayers=6):
    """
    Natrénuje PFN model s danou prior funkcí.
    Přeskočí trénink pokud checkpoint již existuje.
    """
    if os.path.exists(save_path):
        print(f'✓ Checkpoint existuje, přeskakuji: {save_path}')
        return

    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    save_every = 50

    def save_fn(checkpoint, path):
        torch.save(checkpoint, path)
        if checkpoint['epoch'] % save_every == 0:
            torch.save(checkpoint, path.replace('.pth', f'_epoch_{checkpoint["epoch"]}.pth'))

    print(f'Vzorkuji borders ({get_batch_fn.__name__})...')
    ys = get_batch_fn(10000, max_dataset_size, num_features, device='cpu').target_y
    borders = bar_dist_module.get_bucket_borders(num_outputs=1000, ys=ys.cpu()).tolist()

    config = MainConfig(
        train_state_dict_save_path=save_path,
        train_state_dict_load_path=save_path,
        priors=[AdhocPriorConfig(
            get_batch_methods=[get_batch_fn],
            prior_kwargs={'num_features': num_features, 'hyperparameters': {}}
        )],
        optimizer=OptimizerConfig('adamw', lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512, nhead=8, nhid=1024, nlayers=nlayers,
            features_per_group=1, attention_between_features=False,
            encoder=EncoderConfig(constant_normalization_mean=0.5,
                                  constant_normalization_std=math.sqrt(1/12))
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=batch_size, max_seq_len=max_dataset_size,
            min_num_features=num_features, max_num_features=num_features
        ),
        epochs=epochs, warmup_epochs=epochs//4,
        steps_per_epoch=steps_per_epoch, num_workers=0,
    )
    print(f'Trénuji: {nlayers} vrstev, {epochs} epoch, device={device}')
    train(config, device=device, reusable_config=False, save_object_function=save_fn)
    print(f'✓ Uloženo: {save_path}')

print("✓ train_pfn_with_prior připraven")

---
## TRÉNOVÁNÍ — spusť jen jednou (ideálně na GPU/Helios)

Buňky níže trénují tři nové modely. Trénink je automaticky přeskočen pokud checkpoint existuje.

In [ ]:
# Úzký prior (Q4)
train_pfn_with_prior(get_batch_for_gp_narrow_prior,
                     os.path.join('models', 'pfn_narrow_prior_6layer.pth'), epochs=300)

In [ ]:
# Široký prior (Q4)
train_pfn_with_prior(get_batch_for_gp_wide_prior,
                     os.path.join('models', 'pfn_wide_prior_6layer.pth'), epochs=300)

In [ ]:
# Bimodální prior (Q5)
train_pfn_with_prior(get_batch_for_gp_bimodal_prior,
                     os.path.join('models', 'pfn_bimodal_prior_6layer.pth'), epochs=300)

In [ ]:
# =============================================
# NAČTENÍ NATRÉNOVANÝCH MODELŮ
# =============================================

# DŮLEŽITÉ: narrow/wide/bimodal prior funkce musí být definovány před torch.load
# → jsou definovány výše v buňce "PRIOR WRAPPERY"

Q4_MODEL_PATHS = {
    'Úzký prior (σ_ℓ=0.2)':   os.path.join('models', 'pfn_narrow_prior_6layer.pth'),
    'Střední prior (default)': os.path.join('models', 'pfn_rand_hps_6layer.pth'),
    'Široký prior (σ_ℓ=1.5)':  os.path.join('models', 'pfn_wide_prior_6layer.pth'),
}
BIMODAL_PATH = os.path.join('models', 'pfn_bimodal_prior_6layer.pth')

print('Načítám Q4 modely...')
MODELS_Q4 = {}
for name, path in Q4_MODEL_PATHS.items():
    if os.path.exists(path):
        MODELS_Q4[name], _ = load_for_inference(path, device)
    else:
        print(f'  ⚠ Nenalezeno: {path}')

print('\nNačítám bimodální model...')
model_bimodal = None
if os.path.exists(BIMODAL_PATH):
    model_bimodal, _ = load_for_inference(BIMODAL_PATH, device)
else:
    print(f'  ⚠ Nenalezeno: {BIMODAL_PATH}')

reset_seed()
print(f'\nNačteno {len(MODELS_Q4)} Q4 modelů, bimodální: {"✓" if model_bimodal else "✗ chybí"}')

In [ ]:
# =============================================
# Q4: NLL CROSSOVER — funkce
# =============================================
import scipy.stats as stats
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel


def compute_gp_ml_nll(train_x_np, train_y_np, test_x_np, test_y_np):
    """Fituje GP s ML-II a spočítá prediktivní NLL na testovacích bodech."""
    kernel = ConstantKernel(1.0, (1e-3,1e3)) * RBF(0.3, (1e-3,10.)) + WhiteKernel(0.01,(1e-5,1.))
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, normalize_y=False)
    try:
        gp.fit(train_x_np.reshape(-1,1), train_y_np)
        mu, sigma = gp.predict(test_x_np.reshape(-1,1), return_std=True)
        return float(-np.mean(stats.norm.logpdf(test_y_np, mu, np.maximum(sigma, 1e-6))))
    except Exception:
        return np.nan


def compute_pfn_nll(model, train_x, train_y, test_x, test_y, device):
    """Spočítá NLL z BarDistribution výstupu: NLL(y) = -(log P(bin_k) - log width_k)."""
    model.eval()
    with torch.no_grad():
        logits = model(train_x.unsqueeze(0).to(device),
                       train_y.unsqueeze(0).to(device),
                       test_x.unsqueeze(0).to(device))
        borders    = model.criterion.borders.cpu()
        bin_widths = borders[1:] - borders[:-1]
        test_y_cpu = test_y.cpu().squeeze(-1)
        bin_idx    = torch.searchsorted(borders[1:-1].contiguous(),
                                        test_y_cpu.contiguous()).clamp(0, len(bin_widths)-1)
        log_probs  = F.log_softmax(logits[0], dim=-1)
        nll = -(log_probs.gather(-1, bin_idx.unsqueeze(-1)).squeeze(-1)
                - torch.log(bin_widths[bin_idx].clamp(min=1e-10))).mean().item()
    return float(nll)


def run_crossover_experiment(models_dict, n_values, n_test, n_instances, true_hps, device):
    """
    Pro každý model a každé n spočítá:
      delta = NLL_PFN - NLL_GP_ML  (< 0 = PFN vítězí, > 0 = GP-ML vítězí)
    """
    results = {name: {'n': [], 'delta': [], 'pfn_se': []} for name in models_dict}
    for n in n_values:
        print(f'  n={n}: ', end='', flush=True)
        gp_nlls  = []
        pfn_nlls = {name: [] for name in models_dict}
        for _ in range(n_instances):
            batch = get_batch_for_gp(batch_size=1, seq_len=n+n_test, num_features=1,
                                      device='cpu', hyperparameters=true_hps)
            tx, ty = batch.x[0,:n], batch.y[0,:n]
            te_x, te_y = batch.x[0,n:n+n_test], batch.y[0,n:n+n_test]
            gp_nll = compute_gp_ml_nll(tx.numpy().reshape(-1), ty.numpy().reshape(-1),
                                        te_x.numpy().reshape(-1), te_y.numpy().reshape(-1))
            if not np.isnan(gp_nll) and abs(gp_nll) < 100:
                gp_nlls.append(gp_nll)
            for name, model in models_dict.items():
                try:
                    nll = compute_pfn_nll(model, tx, ty, te_x, te_y, device)
                    if not np.isnan(nll) and abs(nll) < 100:
                        pfn_nlls[name].append(nll)
                except Exception:
                    continue
        gp_mean = np.mean(gp_nlls) if gp_nlls else np.nan
        for name in models_dict:
            v = pfn_nlls[name]
            m = np.mean(v) if v else np.nan
            s = np.std(v)/np.sqrt(max(len(v),1)) if v else np.nan
            results[name]['n'].append(n)
            results[name]['delta'].append(m - gp_mean)
            results[name]['pfn_se'].append(s)
            print(f'{name}:Δ={m-gp_mean:.2f}  ', end='', flush=True)
        print()
    return results


def plot_crossover(results):
    fig, ax = plt.subplots(figsize=(9,5))
    colors = plt.cm.tab10.colors
    for i, (name, res) in enumerate(results.items()):
        ax.plot(res['n'], res['delta'], '-o', color=colors[i], lw=2, ms=7, label=name)
        ax.fill_between(res['n'],
                        np.array(res['delta'])-np.array(res['pfn_se']),
                        np.array(res['delta'])+np.array(res['pfn_se']),
                        alpha=0.15, color=colors[i])
    ax.axhline(0, color='black', lw=1.5, ls='--', label='NLL_PFN = NLL_GP-ML (crossover)')
    ax.set_xlabel('Počet tréninkových bodů n', fontsize=12)
    ax.set_ylabel('ΔNLL = NLL_PFN − NLL_GP-ML', fontsize=12)
    ax.set_title('Q4: NLL crossover vs. šíře prioru', fontsize=13)
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

print("✓ Q4 funkce připraveny")

In [ ]:
# =============================================
# SPUŠTĚNÍ Q4
# =============================================

if not MODELS_Q4:
    print("Modely pro Q4 nejsou k dispozici. Spusť tréninkové buňky výše.")
else:
    TRUE_HPS_Q4  = {'lengthscale': 0.3, 'outputscale': 1.0, 'noise': 0.01}
    N_VALUES_Q4  = [2, 5, 10, 20, 50, 100]
    N_TEST_Q4    = 10
    N_INST_Q4    = 50

    reset_seed()
    print('Spouštím Q4: NLL crossover experiment...')
    q4_results = run_crossover_experiment(MODELS_Q4, N_VALUES_Q4, N_TEST_Q4, N_INST_Q4,
                                           TRUE_HPS_Q4, device)
    plot_crossover(q4_results)

    print('\n=== Odhadované crossover n* ===')
    for name, res in q4_results.items():
        deltas = np.array(res['delta'])
        ns     = np.array(res['n'])
        pos    = np.where(deltas > 0)[0]
        if len(pos) > 0:
            print(f'  {name}: n* ≈ {ns[pos[0]]}')
        else:
            print(f'  {name}: crossover nenalezen (PFN vítězí i při n={ns[-1]})')

## Interpretace Q4

**Osa X (log):** počet tréninkových bodů $n$
**Osa Y:** $\Delta\text{NLL} = \text{NLL}_{\text{PFN}} - \text{NLL}_{\text{GP-ML}}$

- $\Delta < 0$: PFN má lepší predikce → PFN vítězí
- $\Delta > 0$: GP-ML vítězí
- Crossover $n^*$: kde $\Delta = 0$

### Predikce z teorie

| Model | Predikce $n^*$ | Důvod |
|---|---|---|
| Úzký prior | Malé $n^*$ | ML-II rychle identifikuje $\ell$ |
| Střední prior | Střední $n^*$ | Kompromis |
| Široký prior | Velké $n^*$ | ML-II potřebuje hodně dat |

PFN výhoda pochází z druhého členu rozptylu směsi:
$\text{Var}[y_*] = \mathbb{E}_\theta[\sigma^2_\theta] + \text{Var}_\theta[\mu_\theta]$
Druhý člen (nejistota o $\theta$) klesá jak $p(\theta|X,y)$ se koncentruje při rostoucím $n$.

In [ ]:
# =============================================
# Q5: BARDISTRIBUTION SHAPE — funkce
# =============================================

def rbf_kernel_q5(x1, x2, ls, scale=1.0):
    return scale * np.exp(-(x1[:,None]-x2[None,:])**2 / (2*ls**2))

def gp_posterior_q5(train_x_np, train_y_np, test_x_val, ls, noise, scale=1.0):
    K   = rbf_kernel_q5(train_x_np, train_x_np, ls, scale)
    Kn  = K + noise*np.eye(len(train_x_np))
    Ks  = rbf_kernel_q5(np.array([test_x_val]), train_x_np, ls, scale)
    Kss = scale
    try:
        L     = np.linalg.cholesky(Kn)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, train_y_np))
        v     = np.linalg.solve(L, Ks.T)
        mu    = float(Ks @ alpha)
        var   = float(Kss - np.sum(v**2))
    except Exception:
        mu, var = 0.0, 1.0
    return mu, np.sqrt(max(var, 1e-8))


def visualize_bardistribution(model, train_x, train_y, test_x_t, test_x_val,
                               ls1=0.1, ls2=0.8, noise=0.01, device='cpu', title=''):
    """
    Vizualizuje BarDistribution výstup PFN vs. GP poster pro oba ℓ a oracle směs.
    """
    model.eval()
    with torch.no_grad():
        logits  = model(train_x.unsqueeze(0).to(device),
                        train_y.unsqueeze(0).to(device),
                        test_x_t.unsqueeze(0).to(device))
        probs   = torch.softmax(logits[0,0], dim=-1).cpu().numpy()
        borders = model.criterion.borders.cpu().numpy()

    bin_widths  = borders[1:] - borders[:-1]
    bin_centers = (borders[:-1] + borders[1:]) / 2
    density     = probs / np.maximum(bin_widths, 1e-10)

    tx_np = train_x.numpy().reshape(-1)
    ty_np = train_y.numpy().reshape(-1)

    y_min = ty_np.min() - 2
    y_max = ty_np.max() + 2
    y_vals = np.linspace(y_min, y_max, 300)
    mask   = (bin_centers >= y_min) & (bin_centers <= y_max)

    mu1, s1 = gp_posterior_q5(tx_np, ty_np, test_x_val, ls1, noise)
    mu2, s2 = gp_posterior_q5(tx_np, ty_np, test_x_val, ls2, noise)
    pdf1    = stats.norm.pdf(y_vals, mu1, s1)
    pdf2    = stats.norm.pdf(y_vals, mu2, s2)
    oracle  = 0.5*pdf1 + 0.5*pdf2

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(bin_centers[mask], density[mask], width=bin_widths[mask],
           alpha=0.55, color='steelblue', label='PFN BarDistribution')
    ax.plot(y_vals, pdf1, 'b--', lw=2, label=f'GP ℓ={ls1} (μ={mu1:.2f}, σ={s1:.2f})')
    ax.plot(y_vals, pdf2, 'g--', lw=2, label=f'GP ℓ={ls2} (μ={mu2:.2f}, σ={s2:.2f})')
    ax.plot(y_vals, oracle, 'r-', lw=2.5, label=f'Oracle směs 0.5·GP_{ls1} + 0.5·GP_{ls2}')
    ax.set_xlim(y_min, y_max)
    ax.set_xlabel('y*', fontsize=12)
    ax.set_ylabel('Hustota pravděpodobnosti', fontsize=12)
    ax.set_title(title or f'Q5: PFN prediktivní distribuce (x*={test_x_val:.1f})', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    pfn_mean = float(np.sum(probs*bin_centers))
    pfn_std  = float(np.sqrt(np.sum(probs*(bin_centers-pfn_mean)**2)))
    print(f'  PFN: mean={pfn_mean:.3f}, std={pfn_std:.3f}')
    print(f'  GP ℓ={ls1}: μ={mu1:.3f}, σ={s1:.3f}')
    print(f'  GP ℓ={ls2}: μ={mu2:.3f}, σ={s2:.3f}')
    print(f'  Oracle: mean={0.5*mu1+0.5*mu2:.3f}')

print("✓ Q5 funkce připraveny")

In [ ]:
# =============================================
# SPUŠTĚNÍ Q5
# =============================================
import scipy.stats as stats

if model_bimodal is None:
    print("Bimodální model není k dispozici. Spusť tréninkovou buňku výše.")
else:
    TEST_X_VAL = 1.5
    N_CONTEXT  = 5

    # Případ 1: Ambiguní kontext — málo bodů, data z ℓ=0.8
    print('=== Případ 1: Ambiguní kontext (n=5, ℓ=0.8) ===')
    batch_amb = get_batch_for_gp(batch_size=1, seq_len=N_CONTEXT+1, num_features=1,
                                  device='cpu', hyperparameters={'lengthscale':0.8,'outputscale':1.0,'noise':0.01})
    tx_amb = batch_amb.x[0, :N_CONTEXT]
    ty_amb = batch_amb.y[0, :N_CONTEXT]
    te_t   = torch.tensor([[TEST_X_VAL]], dtype=torch.float32)
    visualize_bardistribution(model_bimodal, tx_amb, ty_amb, te_t, TEST_X_VAL,
                               device=device, title='Q5: Ambiguní kontext — PFN by měl být bimodální')

    # Případ 2: Jednoznačný kontext — hodně bodů, data z ℓ=0.8
    print('\n=== Případ 2: Jednoznačný kontext (n=20, ℓ=0.8) ===')
    batch_clr = get_batch_for_gp(batch_size=1, seq_len=21, num_features=1,
                                  device='cpu', hyperparameters={'lengthscale':0.8,'outputscale':1.0,'noise':0.01})
    tx_clr = batch_clr.x[0, :20]
    ty_clr = batch_clr.y[0, :20]
    visualize_bardistribution(model_bimodal, tx_clr, ty_clr, te_t, TEST_X_VAL,
                               device=device, title='Q5: Jednoznačný kontext — PFN by měl sledovat ℓ=0.8')

## Interpretace Q5

### Co hledáme

**Ambiguní kontext** (n=5, rovnoměrné rozmístění $X$):
- Data jsou konzistentní jak s $\ell=0.1$, tak s $\ell=0.8$
- PFN output by měl být **heavy-tailed nebo bimodální**

**Jednoznačný kontext** (n=20, data z $\ell=0.8$):
- PFN output by se měl **blížit $\mathcal{N}(\mu_{0.8}, \sigma^2_{0.8})$**

| Pozorování | Závěr |
|---|---|
| PFN ambiguní: bimodální / heavy-tailed | Model zachytil hyperparametrickou nejistotu ✓ |
| PFN ambiguní: úzká Gaussiána | Model se zavázal k jednomu $\ell$ ✗ |
| PFN jednoznačný: blízko GP $\ell=0.8$ | Model identifikoval $\ell$ z kontextu ✓ |

`BarDistribution` umí reprezentovat libovolnou hustotu — pokud trénink fungoval, model by měl tuto kapacitu využít pro ambiguní kontexty.